# Pandas 日练 09 —— 分组聚合（真实业务）

## 业务背景

某电商平台正在分析各区域的订单表现。

数据表中每一行代表一笔订单，包含：

- 区域
- 销售人员
- 订单金额
- 是否退款
- 客户编号

业务部门希望从“订单明细”进一步得到区域层面的经营指标，同时识别高价值区域和异常订单。


In [11]:
import pandas as pd
import numpy as np


df = pd.DataFrame({
    "order_id": [
        1001, 1002, 1003, 1004, 1005,
        1006, 1007, 1008, 1009, 1010,
        1011, 1012
    ],

    "region": [
        "East", "East", "North", "South",
        "East", "North", "South", "East",
        "North", "South", "East", "North"
    ],

    "salesperson": [
        "Alice", "Bob", "Alice", "David",
        "Alice", "Bob", "David", "Bob",
        "Alice", "David", "Alice", "Bob"
    ],

    "customer_id": [
        "C01", "C02", "C03", "C04",
        "C01", "C05", "C06", "C07",
        "C03", "C08", "C09", "C10"
    ],

    "amount": [
        1200, 450, 800, 300,
        1500, 650, 400, 900,
        1100, 500, 700, 750
    ],

    "refunded": [
        False, False, True, False,
        False, False, False, True,
        False, False, False, False
    ]
})

df

,order_id,region,salesperson,customer_id,amount,refunded
0,1001,East,Alice,C01,1200,False
1,1002,East,Bob,C02,450,False
2,1003,North,Alice,C03,800,True
3,1004,South,David,C04,300,False
4,1005,East,Alice,C01,1500,False
5,1006,North,Bob,C05,650,False
6,1007,South,David,C06,400,False
7,1008,East,Bob,C07,900,True
8,1009,North,Alice,C03,1100,False
9,1010,South,David,C08,500,False


## 任务 1：区域基础经营指标

按 `region` 统计每个区域的：

- 订单数量
- 总销售额
- 平均订单金额

最终得到类似：

| region | order_count | total_sales | avg_amount |
|---|---:|---:|---:|

保存为：`region_summary`


In [4]:
region_summary = (
    df
    .groupby(by='region')
    .agg(
        order_count = ('order_id','count'),
        total_sales =('amount','sum'),
        avg_amount = ('amount','mean')
    )
    .reset_index()
)
region_summary

,region,order_count,total_sales,avg_amount
0,East,5,4750,950.0
1,North,4,3300,825.0
2,South,3,1200,400.0


## 任务 2：同时统计多个业务指标

继续按 `region` 统计：

- 订单数量
- 总销售额
- 平均订单金额
- 最大订单金额
- 独立客户数量

要求：

最终结果的字段名清晰，不使用系统自动生成的难以理解的列名。

保存为：`region_metrics`


In [6]:
region_metrics = (
    df
    .groupby(by='region')
    .agg(
        order_count = ('order_id','count'),
        total_sales =('amount','sum'),
        avg_amount = ('amount','mean'),
        max_amount = ('amount','max'),
        total_customers = ('customer_id','nunique')
    )
    .reset_index()
)
region_metrics

,region,order_count,total_sales,avg_amount,max_amount,total_customers
0,East,5,4750,950.0,1500,4
1,North,4,3300,825.0,1100,3
2,South,3,1200,400.0,500,3


## 任务 3：计算每笔订单所在区域的平均订单金额

现在业务人员不想得到区域汇总表，而是希望：

> 保留原来的每一笔订单，同时知道该订单所属区域的平均订单金额。

在原数据基础上新增：`region_avg_amount`


例如：

某条订单属于 East，East 的平均订单金额为 800，那么该订单对应：`region_avg_amount = 800`


要求：

原来的订单明细行数不能减少。

保存为：`df_analysis`


In [10]:
df_analysis = (
    df
    .assign(
        region_avg_amount = lambda x:(
            x.groupby(by='region')['amount']
            .transform('mean')
        )
    )
)
df_analysis

,order_id,region,salesperson,customer_id,amount,refunded,region_avg_amount
0,1001,East,Alice,C01,1200,False,950.0
1,1002,East,Bob,C02,450,False,950.0
2,1003,North,Alice,C03,800,True,825.0
3,1004,South,David,C04,300,False,400.0
4,1005,East,Alice,C01,1500,False,950.0
5,1006,North,Bob,C05,650,False,825.0
6,1007,South,David,C06,400,False,400.0
7,1008,East,Bob,C07,900,True,950.0
8,1009,North,Alice,C03,1100,False,825.0
9,1010,South,David,C08,500,False,400.0


## 任务 4：识别高于区域平均水平的订单

基于任务 3 的结果，新增：`above_region_avg`

规则：

如果：`amount > region_avg_amount`

则：`True`,否则：`False`



In [14]:
df_analysis = (
    df_analysis
    .assign(
        above_region_avg = lambda x:(
            x['amount'] > x['region_avg_amount']
        )
    )
)
df_analysis

,order_id,region,salesperson,customer_id,amount,refunded,region_avg_amount,above_region_avg
0,1001,East,Alice,C01,1200,False,950.0,True
1,1002,East,Bob,C02,450,False,950.0,False
2,1003,North,Alice,C03,800,True,825.0,False
3,1004,South,David,C04,300,False,400.0,False
4,1005,East,Alice,C01,1500,False,950.0,True
5,1006,North,Bob,C05,650,False,825.0,False
6,1007,South,David,C06,400,False,400.0,False
7,1008,East,Bob,C07,900,True,950.0,False
8,1009,North,Alice,C03,1100,False,825.0,True
9,1010,South,David,C08,500,False,400.0,True


## 任务 5：筛选高价值区域

业务部门规定：

> 只分析平均订单金额达到 700 元及以上的区域。

注意：

这里要求的不是生成一张区域汇总表。

而是：

> 把“不符合条件的整个区域”排除掉，同时保留符合条件区域中的全部订单明细。

保存为：`high_value_regions`

In [23]:
high_value_regions = (
    df_analysis
    .query('region_avg_amount >= 700')
        
    )
high_value_regions

,order_id,region,salesperson,customer_id,amount,refunded,region_avg_amount,above_region_avg
0,1001,East,Alice,C01,1200,False,950,True
1,1002,East,Bob,C02,450,False,950,False
2,1003,North,Alice,C03,800,True,825,False
4,1005,East,Alice,C01,1500,False,950,True
5,1006,North,Bob,C05,650,False,825,False
7,1008,East,Bob,C07,900,True,950,False
8,1009,North,Alice,C03,1100,False,825,True
10,1011,East,Alice,C09,700,False,950,False
11,1012,North,Bob,C10,750,False,825,False


## 任务 6：销售人员表现

按照：

- region
- salesperson

两个维度进行统计。

计算每位销售人员的：

- 订单数量
- 总销售额
- 平均订单金额

保存为：

```python
salesperson_summary
```

---


In [26]:
salesperson_summary = (
    df
    .groupby(by=['region','salesperson'])
    .agg(
        order_count = ('order_id','size'),
        total_sales = ('amount','sum'),
        avg_amount = ('amount','mean')
    )
    .reset_index()
)
salesperson_summary

,region,salesperson,order_count,total_sales,avg_amount
0,East,Alice,3,3400,1133.333333
1,East,Bob,2,1350,675.000000
2,North,Alice,2,1900,950.000000
3,North,Bob,2,1400,700.000000
4,South,David,3,1200,400.000000
